# Where does the SFT model hallucinate? (no RAG yet)

RAG costs latency and can pull answers toward adult reading levels (Wikipedia text), so it should be used where
hallucination actually happens. This notebook measures that before any retrieval is added.

- **Part A (Colab, T4)**: the SFT model (config E) answers every knowledge question in the v2 dataset (train,
  validation and test), with token log-probabilities and extra samples kept as candidate uncertainty signals
- **Judge (local)**: `src/judge_hallucination.py` splits each answer into atomic claims and labels each one
  correct / simplified / false / unverifiable, plus the question type and domain
- **Part B (Colab or local, CPU)**: hallucination rate by age, question type, domain, length and truncation;
  whether cheap signals (log-probability, entropy, self-consistency) can flag wrong answers; examples of the errors

The per-question results are reused later: the same questions are answered again with retrieval, and routing
policies are compared offline on these logs.

In [ ]:
!pip install -q -U transformers peft sentence-transformers textstat scikit-learn
!pip uninstall -y -q torchao   # old Colab torchao breaks new PEFT at import time

## 0. Config

In [ ]:
import os, json, gc, re, random
import numpy as np, pandas as pd, torch
import transformers
transformers.logging.set_verbosity_error()

BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DATASET = "yxpan/children_sft_dataset_v2"
SFT_DIR = "/content/drive/MyDrive/llm_sft_runs/sft_lora_v2data"
SFT_RUN = "E_alpha64"            # the SFT model GRPO starts from
EXP_NAME = "hallucination_v2"
SEED = 42
GEN_KWARGS = dict(max_new_tokens=128, do_sample=True, temperature=0.8, top_p=0.9, top_k=30, repetition_penalty=1.05)
GEN_BATCH = 16
K_SAMPLES = 4                    # extra samples per question for the self-consistency signal (0 = skip)
AGES = ["0-3", "4-6", "7-9", "10-13", "14-18"]
SCIENCE = {"physical science", "life science", "earth and space"}   # what the Wikipedia science corpus covers

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = f"/content/drive/MyDrive/llm_eval_runs/{EXP_NAME}"
except ImportError:
    OUT_DIR = os.path.join(os.environ.get("OUT_ROOT", os.path.expanduser("~/llm_eval_runs")), EXP_NAME)
    SFT_DIR = os.environ.get("SFT_DIR", SFT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)
GEN_PATH = f"{OUT_DIR}/generations.jsonl"
JUDGE_PATH = f"{OUT_DIR}/judgments.jsonl"     # written by src/judge_hallucination.py, uploaded here

from datasets import load_dataset
dsd = load_dataset(DATASET)
ROWS = [dict(id=f"{split}-{i}", split=split, age=r["age"], instruction=r["instruction"], reference=r["response"],
             source=r["source"])
        for split in ["train", "validation", "test"] for i, r in enumerate(dsd[split]) if r["needs_knowledge"]]
print(len(ROWS), "knowledge questions;", pd.Series([r["age"] for r in ROWS]).value_counts().reindex(AGES).to_dict())

## Part A. Generate answers with uncertainty signals (GPU)

Same prompt and sampling as the SFT evaluation. For the main answer, the raw (temperature 1) log-probability and
entropy of every generated token are kept; `K_SAMPLES` more answers are sampled for self-consistency.
Results are appended per batch, so a disconnect only loses the current batch.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

def system_prompt(age):
    return (f"You are talking to a child aged {age}. "
            f"Generate a friendly response with age-appropriate knowledge.")

def chat_text(tok, r):
    msgs = [{"role": "system", "content": system_prompt(r["age"])}, {"role": "user", "content": r["instruction"]}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def token_stats(gen_ids, logits, eos_id):
    # gen_ids: (B, T) generated tokens; logits: (B, T, V) raw logits that produced them.
    # Content tokens are those before the first EOS (padding after EOS is also EOS for TinyLlama).
    logp = torch.log_softmax(logits.float(), dim=-1)
    chosen = logp.gather(-1, gen_ids.unsqueeze(-1)).squeeze(-1)
    entropy = -(logp.exp() * logp).sum(-1)
    is_eos = gen_ids == eos_id
    valid = torch.cumsum(is_eos.int(), dim=1) == 0
    out = []
    for b in range(gen_ids.shape[0]):
        v = valid[b]
        n = int(v.sum())
        c, e = chosen[b][v], entropy[b][v]
        out.append(dict(n_tokens=n, truncated=not bool(is_eos[b].any()),
                        mean_logprob=float(c.mean()) if n else None, min_logprob=float(c.min()) if n else None,
                        mean_entropy=float(e.mean()) if n else None))
    return out

def decode(tok, ids):
    return [t.strip() for t in tok.batch_decode(ids, skip_special_tokens=True)]

done = {json.loads(l)["id"] for l in open(GEN_PATH)} if os.path.exists(GEN_PATH) else set()
pending = [r for r in ROWS if r["id"] not in done]
print(f"{len(ROWS) - len(pending)} already generated, {len(pending)} to go")
if pending:
    tok = AutoTokenizer.from_pretrained(BASE_MODEL); tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16)
    model = PeftModel.from_pretrained(base, f"{SFT_DIR}/{SFT_RUN}/adapter").merge_and_unload().to("cuda").eval()
    torch.manual_seed(SEED)
    for i in range(0, len(pending), GEN_BATCH):
        batch = pending[i:i + GEN_BATCH]
        enc = tok([chat_text(tok, r) for r in batch], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        L = enc["input_ids"].shape[1]
        with torch.no_grad():
            out = model.generate(**enc, **GEN_KWARGS, pad_token_id=tok.pad_token_id,
                                 return_dict_in_generate=True, output_logits=True)
            gen = out.sequences[:, L:]
            stats = token_stats(gen, torch.stack(out.logits, dim=1), tok.eos_token_id)
            preds = decode(tok, gen)
            samples = [[] for _ in batch]
            if K_SAMPLES:
                s = model.generate(**enc, **GEN_KWARGS, pad_token_id=tok.pad_token_id, num_return_sequences=K_SAMPLES)
                flat = decode(tok, s[:, L:])
                samples = [flat[j * K_SAMPLES:(j + 1) * K_SAMPLES] for j in range(len(batch))]
        with open(GEN_PATH, "a") as f:
            for r, p, st, sm in zip(batch, preds, stats, samples):
                f.write(json.dumps(dict(r, prediction=p, samples=sm, model=f"sft/{SFT_RUN}", **st)) + "\n")
        print(f"{min(i + GEN_BATCH, len(pending))}/{len(pending)}")
    del model, base; gc.collect(); torch.cuda.empty_cache()
print("generations ->", GEN_PATH)

## Judge (run locally)

Download `generations.jsonl` from `MyDrive/llm_eval_runs/hallucination_v2/`, then from the repo root:

```
python src/judge_hallucination.py --in generations.jsonl --out results/hallucination_v2/judgments.jsonl --limit 5
python src/judge_hallucination.py --in generations.jsonl --out results/hallucination_v2/judgments.jsonl
python src/judge_hallucination.py --in generations.jsonl --out results/hallucination_v2/judgments.jsonl --review 50
```

The first command checks a few answers; the second judges the rest (resumable, stops at the daily quota);
the third writes a CSV of 50 answers' claims for manual labeling (agreement with the judge is computed below).
Upload `judgments.jsonl` (and the filled `judgments_review.csv`, if done) back to the same Drive folder.

## Part B. Where the errors are (CPU)

In [ ]:
import textstat
gen = pd.DataFrame([json.loads(l) for l in open(GEN_PATH)])
jud = pd.DataFrame([json.loads(l) for l in open(JUDGE_PATH)])
D = gen.merge(jud, on="id", how="inner")
print(f"{len(D)} judged of {len(gen)} generated")

NUM = re.compile(r"\b\d[\d,.]*\b|\b(hundred|thousand|million|billion)s?\b", re.I)
def n_names(t):   # capitalized words that do not start a sentence: a rough count of names and places
    return len(re.findall(r"(?<![.!?]\s)(?<!^)\b[A-Z][a-z]+", t))
D["words"] = D.prediction.str.split().str.len()
D["length"] = pd.cut(D.words, [0, 30, 60, 90, 1000], labels=["<=30", "31-60", "61-90", ">90"])
D["has_number"] = D.prediction.str.contains(NUM)
D["has_names"] = D.prediction.map(n_names) > 0
D["in_corpus_domain"] = D.domain.isin(SCIENCE)
D["fkgl"] = D.prediction.map(lambda t: float(textstat.flesch_kincaid_grade(t)) if t.strip() else np.nan)
D["age"] = pd.Categorical(D.age, AGES, ordered=True)
D.to_csv(f"{OUT_DIR}/per_answer.csv", index=False)

has_claims = D[D.n_claims > 0]
print(f"answers with no factual claim: {(D.n_claims == 0).mean():.1%}")
print(f"answers with at least one false claim: {has_claims.has_error.mean():.1%}  "
      f"| false share of checkable claims: {has_claims.false_share.mean():.1%}  "
      f"| simplified claims: {(has_claims.n_simplified.sum() / has_claims.n_claims.sum()):.1%}")

In [ ]:
def slice_table(col):
    g = has_claims.groupby(col, observed=True)
    t = pd.DataFrame(dict(n=g.size(), error_rate=g.has_error.mean(), false_share=g.false_share.mean(),
                          claims=g.n_claims.mean(), simplified=g.n_simplified.sum() / g.n_claims.sum()))
    return t.round(3)

SLICES = {}
for col in ["age", "question_type", "domain", "in_corpus_domain", "length", "truncated", "has_number", "has_names", "split"]:
    SLICES[col] = slice_table(col)
    print(f"\n--- by {col}"); display(SLICES[col])
pd.concat(SLICES, names=["slice", "value"]).to_csv(f"{OUT_DIR}/slices.csv")

import matplotlib.pyplot as plt
pivot = has_claims.pivot_table(index="domain", columns="age", values="has_error", aggfunc="mean", observed=True)
fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(pivot.values, cmap="Reds", vmin=0, vmax=max(0.5, np.nanmax(pivot.values)))
ax.set_xticks(range(len(pivot.columns)), pivot.columns); ax.set_yticks(range(len(pivot.index)), pivot.index)
for (y, x), v in np.ndenumerate(pivot.values):
    if not np.isnan(v): ax.text(x, y, f"{v:.0%}", ha="center", va="center", fontsize=8)
ax.set_title("share of answers with a false claim"); plt.colorbar(im); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/error_by_domain_age.png", dpi=150); plt.show()

### Can a cheap signal flag the wrong answers?

AUROC for predicting "the answer has a false claim" from signals available without retrieval. 0.5 is chance.
Self-consistency: mean embedding similarity between the answer and the extra samples (low = the model says
different things each time).

In [ ]:
from sklearn.metrics import roc_auc_score
from sentence_transformers import SentenceTransformer

H = has_claims.copy()
if K_SAMPLES and H.samples.map(len).gt(0).all():
    emb = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    a = emb.encode(H.prediction.tolist(), normalize_embeddings=True)
    flat = [s for ss in H.samples for s in ss]
    b = emb.encode(flat, normalize_embeddings=True).reshape(len(H), -1, a.shape[1])
    H["self_consistency"] = (b * a[:, None, :]).sum(-1).mean(1)

signals = {"mean log-prob": -H.mean_logprob, "min log-prob": -H.min_logprob, "mean entropy": H.mean_entropy,
           "answer length": H.words}
if "self_consistency" in H:
    signals["self-consistency"] = -H.self_consistency
auroc = {k: roc_auc_score(H.has_error, v.fillna(v.median())) for k, v in signals.items()}
auroc_by_age = {age: {k: (roc_auc_score(H.has_error[m], v[m].fillna(v.median())) if H.has_error[m].nunique() == 2 else np.nan)
                      for k, v in signals.items()} for age in AGES for m in [H.age == age]}
display(pd.Series(auroc, name="AUROC").round(3)); display(pd.DataFrame(auroc_by_age).round(3))
H.drop(columns=["samples", "claims"]).to_csv(f"{OUT_DIR}/per_answer_signals.csv", index=False)

### Error examples and the judge's agreement with manual labels

In [ ]:
false_claims = [(r.age, r.domain, r.instruction, c["claim"], c["note"]) for r in has_claims.itertuples()
                for c in r.claims if c["verdict"] == "false"]
print(len(false_claims), "false claims")
for age, dom, q, c, note in random.Random(SEED).sample(false_claims, min(15, len(false_claims))):
    print(f"[{age} | {dom}] {q[:70]}\n   claim: {c}\n   why  : {note}")

review = JUDGE_PATH.replace(".jsonl", "_review.csv")
if os.path.exists(review):
    from sklearn.metrics import cohen_kappa_score
    rv = pd.read_csv(review).dropna(subset=["human_verdict"])
    to_err = lambda s: s.str.strip().eq("false")
    print(f"\n{len(rv)} manually labeled claims | verdict agreement {np.mean(rv.judge_verdict == rv.human_verdict.str.strip()):.1%}"
          f" | kappa on false vs not false {cohen_kappa_score(to_err(rv.judge_verdict), to_err(rv.human_verdict)):.3f}")

In [ ]:
summary = dict(
    n_answers=int(len(D)), share_no_claims=float((D.n_claims == 0).mean()),
    error_rate=float(has_claims.has_error.mean()), false_share=float(has_claims.false_share.mean()),
    error_rate_by_age=SLICES["age"].error_rate.to_dict(),
    error_rate_by_question_type=SLICES["question_type"].error_rate.to_dict(),
    error_rate_in_vs_out_of_corpus_domain={str(k): v for k, v in SLICES["in_corpus_domain"].error_rate.to_dict().items()},
    auroc=auroc, judge_model=str(jud.judge_model.iloc[0]),
)
json.dump(summary, open(f"{OUT_DIR}/summary.json", "w"), indent=2)
summary